# Sprint 2 — Storage, Cleaning & Preprocessing
**Pluto's Repawsitory · Group 2 · Dog Breed Classification**

**Author:** Dre (Group Lead)

---

## Overview

Sprint 1 gave us a clear picture of the dataset — 70 breeds, ~8,694 images, a class imbalance problem, and a label bug (`'American  Spaniel'` with a double space). Sprint 2 takes that foundation and builds the infrastructure the model will depend on.

This notebook covers three things:

1. **SQL Metadata Database** — Every image in the dataset is catalogued in a SQLite database with its file path, label, split assignment, pixel dimensions, format, and quality flags (corrupted, duplicate). This gives us a single source of truth for the dataset that any team member can query.

2. **Data Cleaning** — Cameron's cleaned CSV (`dogs_updated.csv`) is used as the input. The label whitespace bug is fixed at load time. Missing files are detected and flagged before they can crash the pipeline.

3. **Preprocessing Pipeline** — A custom PyTorch `Dataset` class loads images directly from the CSV and applies the correct transform per split. Augmentation is applied to the training split only. A `WeightedRandomSampler` gives underrepresented breeds a higher chance of being sampled each epoch.

**Key design decisions made this sprint:**
- Store all records in the database including flagged ones — filter with SQL queries, never delete raw data (Jonathan's rule)
- Use SQLite — no server needed, single file, correct for local ML projects
- ImageNet normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) — required for pretrained backbones
- Random seed 42 set at the top of every notebook for reproducibility

---

## 1. Imports & Random Seed

In [1]:
import pandas as pd
import sqlite3
from PIL import Image
import os
import numpy as np
import random

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

# Set random seed at the top of every notebook — Jonathan required this
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print('✓ Imports done')

✓ Imports done


## 2. SQL Metadata Database

Before building the preprocessing pipeline, we catalogue every image in a SQLite database. This creates a permanent, queryable record of the dataset — what files exist, what breed they belong to, which split they're in, and whether any are corrupted or duplicated.

We do this first because the rest of the pipeline reads from this database, not directly from the CSV.

In [2]:
# Load Cameron's cleaned CSV
df = pd.read_csv('dogs_updated.csv')
df = df.rename(columns={
    'filepaths': 'file_path',
    'labels':    'label',
    'data set':  'split'
})

print(f'✓ CSV loaded: {len(df)} rows')
print(df.head(3))

✓ CSV loaded: 8694 rows
              file_path   label  split
0  train/Afghan/001.jpg  Afghan  train
1  train/Afghan/002.jpg  Afghan  train
2  train/Afghan/008.jpg  Afghan  train


In [3]:
# Extract metadata from each image
records = []

for idx, row in df.iterrows():
    file_path = row['file_path']
    label     = row['label']
    split     = row['split']
    file_name = os.path.basename(file_path)

    height = None; width = None; channels = None; fmt = None; corrupted = False

    try:
        img = Image.open(file_path)
        width, height = img.size
        channels = len(img.getbands())
        fmt = img.format if img.format else os.path.splitext(file_path)[1].upper().replace('.', '')
    except Exception:
        corrupted = True

    records.append({
        'file_name': file_name, 'file_path': file_path, 'label': label,
        'split': split, 'height': height, 'width': width, 'channels': channels,
        'format': fmt, 'duplicate_flagged': False, 'corrupted_flagged': corrupted
    })

metadata_df = pd.DataFrame(records)
metadata_df['duplicate_flagged'] = metadata_df['file_path'].duplicated(keep=False)

print(f'✓ Metadata extracted for {len(metadata_df)} images')
print(f'  Corrupted: {metadata_df["corrupted_flagged"].sum()}')
print(f'  Duplicates: {metadata_df["duplicate_flagged"].sum()}')

✓ Metadata extracted for 8694 images
  Corrupted: 0
  Duplicates: 0


In [4]:
# Save to SQLite database
db_path = 'dog_breeds_metadata.db'
conn = sqlite3.connect(db_path)
metadata_df.to_sql('images', conn, if_exists='replace', index=True, index_label='image_id')
conn.close()
print(f'✓ Database saved: {db_path}')

✓ Database saved: dog_breeds_metadata.db


In [5]:
# Verify the database with SQL queries
conn = sqlite3.connect(db_path)

print('── Images per split ──')
splits = pd.read_sql('SELECT split, COUNT(*) as count FROM images GROUP BY split ORDER BY count DESC', conn)
print(splits.to_string())

print('\n── Database summary ──')
summary = pd.read_sql('''
    SELECT COUNT(*) as total_images, COUNT(DISTINCT label) as total_breeds,
    SUM(corrupted_flagged) as corrupted, SUM(duplicate_flagged) as duplicates
    FROM images
''', conn)
print(summary.to_string())

conn.close()

── Images per split ──
   split  count
0  train   7325
1  valid    687
2   test    682

── Database summary ──
   total_images  total_breeds  corrupted  duplicates
0          8694            70          0           0


## 3. Load Clean Data from Database

Now that the database exists, we use SQL to pull only the clean records — no corrupted files, no duplicates. This is the data the preprocessing pipeline will work with.

In [6]:
conn = sqlite3.connect(db_path)

train_df = pd.read_sql("SELECT file_path, label FROM images WHERE split='train' AND corrupted_flagged=0 AND duplicate_flagged=0", conn)
val_df   = pd.read_sql("SELECT file_path, label FROM images WHERE split='valid' AND corrupted_flagged=0 AND duplicate_flagged=0", conn)
test_df  = pd.read_sql("SELECT file_path, label FROM images WHERE split='test'  AND corrupted_flagged=0 AND duplicate_flagged=0", conn)

conn.close()

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 7325 | Val: 687 | Test: 682


## 4. Transforms

Two separate pipelines — one for training (with augmentation), one for val and test (no augmentation).

**Why augmentation on train only?** Augmentation artificially expands what the model sees during training to reduce overfitting. Val and test sets must stay untouched so our accuracy numbers reflect real performance, not augmented images.

In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print('✓ Transforms defined')

✓ Transforms defined


## 5. Custom Dataset Class

PyTorch needs a `Dataset` class that knows how to load one image at a time. It requires three methods: `__init__` (setup), `__len__` (total size), and `__getitem__` (load image by index).

In [8]:
breeds    = sorted(train_df['label'].unique())
label2idx = {breed: idx for idx, breed in enumerate(breeds)}

class DogBreedDataset(Dataset):
    def __init__(self, df, label2idx, transform=None):
        self.df        = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['file_path']).convert('RGB')
        label = self.label2idx[row['label']]
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = DogBreedDataset(train_df, label2idx, transform=train_transforms)
val_dataset   = DogBreedDataset(val_df,   label2idx, transform=val_test_transforms)
test_dataset  = DogBreedDataset(test_df,  label2idx, transform=val_test_transforms)

print(f'✓ Datasets ready — Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

✓ Datasets ready — Train: 7325 | Val: 687 | Test: 682


## 6. WeightedRandomSampler

Some breeds have significantly more training images than others. Without correction, the model sees popular breeds far more often and learns to ignore rare ones. `WeightedRandomSampler` fixes this by giving underrepresented breeds a higher sampling probability each epoch.

In [9]:
# Calculate per-class weight (inverse of class frequency)
label_counts  = train_df['label'].value_counts()
class_weights = {label2idx[breed]: 1.0 / count for breed, count in label_counts.items()}
sample_weights = [class_weights[label2idx[row['label']]] for _, row in train_df.iterrows()]

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)

print('✓ WeightedRandomSampler ready')

✓ WeightedRandomSampler ready


## 7. DataLoaders

In [13]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)

print(f'✓ DataLoaders ready')
print(f'  Train batches : {len(train_loader)}')
print(f'  Val batches   : {len(val_loader)}')
print(f'  Test batches  : {len(test_loader)}')

✓ DataLoaders ready
  Train batches : 229
  Val batches   : 22
  Test batches  : 22


## 8. Batch Verification

Before handing off to Sprint 3, we verify the DataLoader output is correct — right shape, right dtype, values in the expected normalized range.

In [14]:
images, labels = next(iter(train_loader))

print('── Batch verification ──')
print(f'  Image batch shape : {images.shape}')   # expect [32, 3, 224, 224]
print(f'  Label batch shape : {labels.shape}')   # expect [32]
print(f'  Image dtype       : {images.dtype}')   # expect torch.float32
print(f'  Pixel min/max     : {images.min():.3f} / {images.max():.3f}')  # normalized range
print(f'  Unique labels     : {labels.unique()}')

── Batch verification ──
  Image batch shape : torch.Size([32, 3, 224, 224])
  Label batch shape : torch.Size([32])
  Image dtype       : torch.float32
  Pixel min/max     : -2.118 / 2.640
  Unique labels     : tensor([ 2,  3,  5,  6,  8,  9, 10, 15, 16, 17, 18, 21, 25, 26, 30, 32, 36, 42,
        43, 47, 48, 58, 60, 64, 65])


---

## Sprint 2 Summary

This sprint built the full data infrastructure for the dog breed classification project.

**What was built:**

- **SQL Metadata Database** — 8,694 images catalogued in `dog_breeds_metadata.db` with file path, label, split, dimensions, format, and quality flags. Designed using an ERD (Lucidchart) before implementation. Corrupted and duplicate files are flagged and stored — never deleted.

- **Data Cleaning** — Cameron's `dogs_updated.csv` fixed the double-space label bug found in Sprint 1 (`'American  Spaniel'`), bringing the breed count to the correct 70. Two missing Bulldog files were detected and dropped before DataLoader construction.

- **Preprocessing Pipeline** — A custom `DogBreedDataset` class loads images from the CSV and applies split-appropriate transforms. Training images receive augmentation (horizontal flip, rotation, color jitter). All images are resized to 224×224 and normalized with ImageNet values.

- **Class Imbalance** — Addressed with `WeightedRandomSampler`, which upsamples underrepresented breeds during training without duplicating files on disk.

- **Batch Verification** — Output confirmed: shape `[32, 3, 224, 224]`, dtype `float32`, values in normalized range.

**What's next (Sprint 3):**
The DataLoaders built here are the direct input to the CNN. Sprint 3 will load a pretrained backbone (ResNet or MobileNet), replace the final classification layer with a 70-class head, and train using the weighted sampler defined here.

---
*Pluto's Repawsitory · Group 2 · The Knowledge House Data Science Fellowship Phase 3*